# AlphaBuilder v3.3 - Kaggle Distributed Training (`train_pipeline.py`)

Notebook oficial de treinamento supervisionado **AlphaBuilder v3.3** (Bloco B) para **Kaggle GPU (`T4 x2` / `T4`)**:

- **`D4 x Z2 = 16x` simetrias 3D** (8x flips no `TopologyDatasetV31` + 2x `rot90` por batch)
- **GroupNorm + `phys_film` + CoordConv** preservando escalas físicas de força e contorno (`FULL_CLAMP` / `RAIL_XY`)
- **Multi-Head Loss calibrada v3.3**: masked KL Policy (`lambda=1.0`) + Huber Value (`lambda=2.0`) + Blueprint BCE/Dice (`lambda_bp=0.50`)
- **DDP real multi-GPU** via `torchrun --standalone --nproc_per_node=2` sobre `warmup_v3_3_kaggle.db` e `split_v3_3.json`

**Presets disponíveis em `train_config.py`:**
- **Etapa 3 (Baseline rápido):** `PRESET = "kaggle_t4_dual_simple"` (`SimpleBackbone`, RF >= 64, per-GPU BS=64)
- **Etapa 5 (Modelo principal):** `PRESET = "kaggle_t4_dual_swin"` (`Swin-UNETR`, per-GPU BS=8)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import torch

print("=" * 64)
print("AlphaBuilder v3.3 - Kaggle Training Pipeline")
print("=" * 64)
print(f"CUDA available: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}")
for idx in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(idx)
    print(f"  GPU {idx}: {torch.cuda.get_device_name(idx)} ({props.total_memory / 1024**3:.1f} GB)")


In [ ]:
# === Configuracao da Execucao ===
# Etapa 3: "kaggle_t4_dual_simple"
# Etapa 5: "kaggle_t4_dual_swin"
PRESET = "kaggle_t4_dual_simple"
EPOCHS_OVERRIDE = None      # Ex.: 40 ou None para usar o default do preset
LAMBDA_BP_OVERRIDE = None   # Ex.: 0.50 ou 0.25, None usa default do preset
RESUME_CHECKPOINT = None    # Ex.: "/kaggle/working/checkpoints/last_model.pt"

REPO_URL = "https://github.com/gustavomello9600/alphabuild.git"
REPO_DIR = Path("/kaggle/working/alphabuild")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

if "swin" in PRESET:
    try:
        import monai  # noqa: F401
        print("MONAI already installed:", monai.__version__)
    except ImportError:
        print("Installing MONAI for Swin-UNETR...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai"], check=True)


In [ ]:
# === Descoberta Automatica de warmup_v3_3_kaggle.db e split_v3_3.json ===
search_roots = [Path("/kaggle/input"), REPO_DIR / "data" / "compiled", Path("/kaggle/working")]

db_candidates = []
split_candidates = []
for root in search_roots:
    if root.exists():
        db_candidates.extend(sorted(root.rglob("warmup_v3_3_kaggle.db")))
        split_candidates.extend(sorted(root.rglob("split_v3_3.json")))

if not db_candidates:
    raise FileNotFoundError(
        "warmup_v3_3_kaggle.db nao encontrado em /kaggle/input! "
        "Anexe o Dataset Kaggle contendo warmup_v3_3_kaggle.db e split_v3_3.json ao notebook."
    )

DB_PATH = db_candidates[0]
SPLIT_PATH = split_candidates[0] if split_candidates else None

print(f"Database: {DB_PATH} ({DB_PATH.stat().st_size / 1e6:.2f} MB)")
print(f"Split:    {SPLIT_PATH}")


In [ ]:
# === Treinamento Multi-GPU (DDP via torchrun) ou Single-GPU ===
gpu_count = torch.cuda.device_count()
if gpu_count <= 1 and "dual" in PRESET:
    PRESET = PRESET.replace("_dual_", "_")
    print(f"Single GPU detectada -> ajustando preset para {PRESET}")

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
LOG_DIR = Path("/kaggle/working/logs")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

if gpu_count > 1:
    cmd = [
        "torchrun",
        "--standalone",
        f"--nproc_per_node={gpu_count}",
        "alphabuilder/src/neural/train_pipeline.py",
    ]
else:
    cmd = [sys.executable, "alphabuilder/src/neural/train_pipeline.py"]

cmd.extend([
    "--config", PRESET,
    "--db-path", str(DB_PATH),
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--log-dir", str(LOG_DIR),
])
if SPLIT_PATH is not None:
    cmd.extend(["--split-json", str(SPLIT_PATH)])
if EPOCHS_OVERRIDE is not None:
    cmd.extend(["--epochs", str(EPOCHS_OVERRIDE)])
if LAMBDA_BP_OVERRIDE is not None:
    cmd.extend(["--lambda-bp", str(LAMBDA_BP_OVERRIDE)])
if RESUME_CHECKPOINT:
    cmd.extend(["--resume", str(RESUME_CHECKPOINT)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# === Resumo, Curvas de Loss e Exportacao para /kaggle/working ===
import json
import shutil

history_path = CHECKPOINT_DIR / "metrics_history.json"
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    if history:
        best_ep = min(history, key=lambda h: h["val"].get("loss", float("inf")))
        print(f"Melhor epoca: {best_ep['epoch']} | Val Loss={best_ep['val']['loss']:.4f} | "
              f"Top1={best_ep['val'].get('policy_top1_acc', 0.0):.3f} | "
              f"MAE={best_ep['val'].get('value_mae', 0.0):.4f} | "
              f"Dice={best_ep['val'].get('blueprint_dice', 0.0):.3f}")
    try:
        import matplotlib.pyplot as plt
        epochs = [h["epoch"] for h in history]
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].plot(epochs, [h["train"]["loss"] for h in history], label="Train")
        axes[0].plot(epochs, [h["val"]["loss"] for h in history], label="Val")
        axes[0].set_title("Total Loss")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, [h["val"].get("policy_top1_acc", 0.0) for h in history], label="Top-1 Acc")
        axes[1].plot(epochs, [h["val"].get("blueprint_dice", 0.0) for h in history], label="Blueprint Dice")
        axes[1].set_title("Policy Top-1 & Blueprint Dice")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(epochs, [h["val"].get("value_mae", 0.0) for h in history], label="Value MAE")
        axes[2].set_title("Value MAE (Val)")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("/kaggle/working/training_curves.png", dpi=150)
        plt.show()
    except Exception as exc:
        print("Plot skipped:", exc)

for fname in ("best_model.pt", "last_model.pt", "metrics_history.json"):
    src = CHECKPOINT_DIR / fname
    if src.exists():
        shutil.copy2(src, Path("/kaggle/working") / fname)
        print(f"Exported: /kaggle/working/{fname} ({src.stat().st_size / 1e6:.2f} MB)")
